SPLIT THEM TO 3 FOLDERS

In [ ]:
from pathlib import Path
import shutil, json, csv, random, re
from collections import defaultdict
from datetime import datetime

# =========================
# CONFIG
# =========================
DATA_INTERIM = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_512_noborder_balanced")

OUT_PROCESSED = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
OUT_IMAGES = OUT_PROCESSED / "images"
OUT_INDEX  = OUT_PROCESSED / "index.csv"
OUT_SPLITS = OUT_PROCESSED / "splits.json"

TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
SEED = 42

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}

# =========================
# HELPERS
# =========================
def extract_timestamp(filename: str) -> str:
    m = re.search(r"(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})-(\d{3})", filename)
    if not m:
        return ""
    return f"{m.group(1)}T{m.group(2).replace('-', ':')}.{m.group(3)}"

def list_images(root: Path):
    return sorted([
        p for p in root.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])

def base_group_id(filename: str) -> str:
    """
    Keep original + augmented versions together.
    Example:
    xxx_orig.png and xxx_aug01.png will not be split apart.
    """
    m = re.match(r"(.+?)_(orig|balflip\d+)\.[^.]+$", filename)
    if m:
        return m.group(1)
    return Path(filename).stem

def split_grouped_images(image_paths, seed=42):
    groups = defaultdict(list)

    for p in image_paths:
        gid = base_group_id(p.name)
        groups[gid].append(p)

    group_ids = sorted(groups.keys())
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    n_groups = len(group_ids)
    n_train = int(n_groups * TRAIN_RATIO)
    n_val = int(n_groups * VAL_RATIO)

    train_g = set(group_ids[:n_train])
    val_g = set(group_ids[n_train:n_train + n_val])
    test_g = set(group_ids[n_train + n_val:])

    rows = []

    for gid, plist in groups.items():
        if gid in train_g:
            split = "train"
        elif gid in val_g:
            split = "val"
        else:
            split = "test"

        for p in plist:
            rows.append({
                "split": split,
                "src_path": str(p),
                "filename": p.name,
                "group_id": gid,
                "scan_timestamp": extract_timestamp(p.name),
            })

    return rows

def write_outputs(rows):
    for sp in ("train", "val", "test"):
        (OUT_IMAGES / sp).mkdir(parents=True, exist_ok=True)

    final_rows = []
    splits_json = {"train": [], "val": [], "test": []}

    for r in rows:
        split = r["split"]
        src = Path(r["src_path"])

        new_name = src.name
        dst_rel = f"images/{split}/{new_name}"
        dst_abs = OUT_PROCESSED / dst_rel

        shutil.copy2(src, dst_abs)

        final_rows.append({
            "filepath": dst_rel,
            "split": split,
            "filename": new_name,
            "scan_timestamp": r["scan_timestamp"],
            "source_interim_path": str(src),
            "group_id": r["group_id"],
        })

        splits_json[split].append(dst_rel)

    with open(OUT_SPLITS, "w") as f:
        json.dump({
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "source": str(DATA_INTERIM),
            "seed": SEED,
            "ratios": {
                "train": TRAIN_RATIO,
                "val": VAL_RATIO,
                "test": TEST_RATIO,
            },
            "counts": {k: len(v) for k, v in splits_json.items()},
            "splits": splits_json,
        }, f, indent=2)

    with open(OUT_INDEX, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
        w.writeheader()
        w.writerows(final_rows)

    return final_rows, splits_json

# =========================
# RUN
# =========================
OUT_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Listing images...")
image_paths = list_images(DATA_INTERIM)
print(f"Found {len(image_paths)} images")

rows = split_grouped_images(image_paths, seed=SEED)
final_rows, splits_json = write_outputs(rows)

print("Saved processed dataset to:", OUT_PROCESSED)
print("Split counts:", {k: len(v) for k, v in splits_json.items()})
print("Index saved to:", OUT_INDEX)
print("Splits saved to:", OUT_SPLITS)

Automate Labelling 

In [ ]:
from pathlib import Path
import shutil, json, csv, random, re
from collections import defaultdict, Counter
from datetime import datetime

# =========================
# CONFIG
# =========================
DATA_INTERIM = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_512_noborder_balanced")

# Label Studio exported JSON file path
LABEL_STUDIO_JSON = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/result_labels.json")

OUT_PROCESSED = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
OUT_IMAGES = OUT_PROCESSED / "images"
OUT_INDEX  = OUT_PROCESSED / "index.csv"
OUT_SPLITS = OUT_PROCESSED / "splits.json"

OUT_LABELS_DIR = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
OUT_LABELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TRAIN_LBL = OUT_LABELS_DIR / "train.json"
OUT_VAL_LBL   = OUT_LABELS_DIR / "val.json"
OUT_TEST_LBL  = OUT_LABELS_DIR / "test.json"

TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
SEED = 42

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}

# Change these only if your Label Studio choice names are different
LABEL_KEYS = [
    "Empty",
    "Non-Empty",
    "Overlap",
    "Isolated",
    "Contraband",
    "Non-Contraband",
]

# Extra non-contraband overlap images to add
EXTRA_NONCONTRABAND_OVERLAP_DIR = Path("../../data/interim/Stage2/gray_clahe_1500x1000_noborder_aug")

# Auto labels for newly added images
EXTRA_LABELS = {
    "Empty": 0,
    "Non-Empty": 1,
    "Overlap": 1,
    "Isolated": 0,
    "Contraband": 0,
    "Non-Contraband": 1,
}

# Add only enough to balance approximately
ENABLE_EXTRA_BALANCING = True

# =========================
# HELPERS
# =========================
def list_images(root: Path):
    return sorted([
        p for p in root.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])

def add_extra_noncontraband_overlap_images(image_paths, labels_by_filename, seed=42):
    """
    Adds only enough extra non-contraband overlap images to improve balance.

    Extra images are auto-labelled as:
    non_empty=1, overlap=1, non_contraband=1
    """

    if not ENABLE_EXTRA_BALANCING:
        return image_paths, labels_by_filename

    if not EXTRA_NONCONTRABAND_OVERLAP_DIR.exists():
        print("Extra folder not found:", EXTRA_NONCONTRABAND_OVERLAP_DIR)
        return image_paths, labels_by_filename

    extra_paths = list_images(EXTRA_NONCONTRABAND_OVERLAP_DIR)
    if len(extra_paths) == 0:
        print("No extra images found in:", EXTRA_NONCONTRABAND_OVERLAP_DIR)
        return image_paths, labels_by_filename

    # Current label counts
    current_counts = Counter()

    for p in image_paths:
        labels = labels_by_filename.get(p.name)
        if labels is None:
            continue

        for k, v in labels.items():
            if v == 1:
                current_counts[k] += 1

    overlap_count = current_counts["Overlap"]
    isolated_count = current_counts["Isolated"]
    contraband_count = current_counts["Contraband"]
    non_contraband_count = current_counts["Non-Contraband"]

    # Need enough extra images to balance both:
    # overlap vs isolated
    # contraband vs non-contraband
    need_for_overlap = max(0, isolated_count - overlap_count)
    need_for_non_contraband = max(0, contraband_count - non_contraband_count)

    num_to_add = min(
        max(need_for_overlap, need_for_non_contraband),
        len(extra_paths)
    )

    print("\nExtra balancing calculation:")
    print("  Current overlap:", overlap_count)
    print("  Current isolated:", isolated_count)
    print("  Current contraband:", contraband_count)
    print("  Current non_contraband:", non_contraband_count)
    print("  Need for overlap balance:", need_for_overlap)
    print("  Need for non_contraband balance:", need_for_non_contraband)
    print("  Extra available:", len(extra_paths))
    print("  Extra selected:", num_to_add)

    rng = random.Random(seed)
    rng.shuffle(extra_paths)
    selected_extra = extra_paths[:num_to_add]

    # Auto-label selected extra images
    for p in selected_extra:
        labels_by_filename[p.name] = dict(EXTRA_LABELS)

    return image_paths + selected_extra, labels_by_filename

def base_group_id(filename: str) -> str:
    """
    Keep original + augmented versions together.
    """
    m = re.match(r"(.+?)_(orig|balflip\d+)\.[^.]+$", filename)
    if m:
        return m.group(1)
    return Path(filename).stem

def extract_timestamp(filename: str) -> str:
    m = re.search(r"(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})-(\d{3})", filename)
    if not m:
        return ""
    return f"{m.group(1)}T{m.group(2).replace('-', ':')}.{m.group(3)}"

def load_label_studio_labels(json_path: Path):
    """
    Supports balanced JSON format:
    [
      {
        "image": "xxx_orig.png",
        "labels": ["contraband", "overlap"]
      }
    ]
    """

    data = json.load(open(json_path, "r"))

    labels_by_filename = {}

    for item in data:
        if not isinstance(item, dict):
            continue

        fname = Path(item.get("image", "")).name
        raw_labels = item.get("labels", [])

        labels = {k: 0 for k in LABEL_KEYS}

        norm = set()
        for x in raw_labels:
            x = str(x).lower().strip().replace("-", "_").replace(" ", "_")
            norm.add(x)

        if "empty" in norm:
            labels["Empty"] = 1
        if "non_empty" in norm or "nonempty" in norm:
            labels["Non-Empty"] = 1
        if "overlap" in norm:
            labels["Overlap"] = 1
        if "isolated" in norm:
            labels["Isolated"] = 1
        if "contraband" in norm:
            labels["Contraband"] = 1
        if "non_contraband" in norm or "noncontraband" in norm:
            labels["Non-Contraband"] = 1

        if fname:
            labels_by_filename[fname] = labels

    return labels_by_filename

def split_grouped_images(image_paths, seed=42):
    groups = defaultdict(list)

    for p in image_paths:
        gid = base_group_id(p.name)
        groups[gid].append(p)

    group_ids = sorted(groups.keys())
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    n_groups = len(group_ids)
    n_train = int(n_groups * TRAIN_RATIO)
    n_val = int(n_groups * VAL_RATIO)

    train_g = set(group_ids[:n_train])
    val_g = set(group_ids[n_train:n_train + n_val])
    test_g = set(group_ids[n_train + n_val:])

    rows = []

    for gid, plist in groups.items():
        if gid in train_g:
            split = "train"
        elif gid in val_g:
            split = "val"
        else:
            split = "test"

        for p in plist:
            rows.append({
                "split": split,
                "src_path": str(p),
                "filename": p.name,
                "group_id": gid,
                "scan_timestamp": extract_timestamp(p.name),
            })

    return rows

def write_outputs(rows, labels_by_filename, polygons_by_filename):
    for sp in ("train", "val", "test"):
        (OUT_IMAGES / sp).mkdir(parents=True, exist_ok=True)
        (OUT_MASKS / sp).mkdir(parents=True, exist_ok=True)

    final_rows = []
    splits_json = {"train": [], "val": [], "test": []}
    label_items = {"train": [], "val": [], "test": []}

    missing_labels = []

    for r in rows:
        split = r["split"]
        src = Path(r["src_path"])

        new_name = src.name
        dst_rel = f"images/{split}/{new_name}"
        dst_abs = OUT_PROCESSED / dst_rel

        shutil.copy2(src, dst_abs)

        mask_rel = f"masks/{split}/{new_name}"
        mask_abs = OUT_PROCESSED / mask_rel

        polygon_key = src.name

        if polygon_key not in polygons_by_filename:
            original_name = base_group_id(src.name) + src.suffix
            polygon_key = original_name

        if polygon_key in polygons_by_filename:
            mask_img = create_polygon_mask(src, polygons_by_filename[polygon_key])
        else:
            img_tmp = Image.open(src).convert("L")
            mask_img = Image.new("L", img_tmp.size, 0)

        mask_img.save(mask_abs)

        labels = labels_by_filename.get(src.name)

        if labels is None:
            original_name = base_group_id(src.name) + src.suffix
            labels = labels_by_filename.get(original_name)

        if labels is None:
            labels = {k: 0 for k in LABEL_KEYS}
            missing_labels.append(src.name)

        final_rows.append({
            "filepath": dst_rel,
            "split": split,
            "filename": new_name,
            "scan_timestamp": r["scan_timestamp"],
            "source_interim_path": str(src),
            "group_id": r["group_id"],

            "empty": labels["Empty"],
            "non_empty": labels["Non-Empty"],
            "overlap": labels["Overlap"],
            "isolated": labels["Isolated"],
            "contraband": labels["Contraband"],
            "non_contraband": labels["Non-Contraband"],
        })

        splits_json[split].append(dst_rel)

        label_items[split].append({
            "image": dst_rel,
            "empty": labels["Empty"],
            "non_empty": labels["Non-Empty"],
            "overlap": labels["Overlap"],
            "isolated": labels["Isolated"],
            "contraband": labels["Contraband"],
            "non_contraband": labels["Non-Contraband"],
        })

    with open(OUT_SPLITS, "w") as f:
        json.dump({
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "source_images": str(DATA_INTERIM),
            "source_labels": str(LABEL_STUDIO_JSON),
            "seed": SEED,
            "ratios": {
                "train": TRAIN_RATIO,
                "val": VAL_RATIO,
                "test": TEST_RATIO,
            },
            "counts": {k: len(v) for k, v in splits_json.items()},
            "missing_label_count": len(missing_labels),
            "missing_labels": missing_labels[:50],
            "splits": splits_json,
        }, f, indent=2)

    with open(OUT_INDEX, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
        w.writeheader()
        w.writerows(final_rows)

    json.dump(label_items["train"], open(OUT_TRAIN_LBL, "w"), indent=2)
    json.dump(label_items["val"], open(OUT_VAL_LBL, "w"), indent=2)
    json.dump(label_items["test"], open(OUT_TEST_LBL, "w"), indent=2)

    return final_rows, splits_json, label_items, missing_labels

# =========================
# RUN
# =========================
OUT_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Loading Label Studio JSON...")
labels_by_filename, polygons_by_filename = load_label_studio_labels(LABEL_STUDIO_JSON)
print("Loaded labels for:", len(labels_by_filename), "images")

print("Listing images...")
image_paths = list_images(DATA_INTERIM)
print("Found original images:", len(image_paths))

image_paths, labels_by_filename = add_extra_noncontraband_overlap_images(
    image_paths,
    labels_by_filename,
    seed=SEED,
)

print("Found images after extra balancing:", len(image_paths))

rows = split_grouped_images(image_paths, seed=SEED)
final_rows, splits_json, label_items, missing_labels = write_outputs(rows, labels_by_filename)

print("Saved processed dataset to:", OUT_PROCESSED)
print("Index saved to:", OUT_INDEX)
print("Splits saved to:", OUT_SPLITS)

print("Label files:")
print(" ", OUT_TRAIN_LBL)
print(" ", OUT_VAL_LBL)
print(" ", OUT_TEST_LBL)

print("Split counts:", {k: len(v) for k, v in splits_json.items()})
print("Missing label count:", len(missing_labels))

for split in ("train", "val", "test"):
    print(f"\n{split} label distribution:")
    for key in ["empty", "non_empty", "overlap", "isolated", "contraband", "non_contraband"]:
        print(f"  {key}: {sum(x[key] for x in label_items[split])}")

In [ ]:
from pathlib import Path
import shutil, json, csv, random, re
from collections import defaultdict, Counter
from datetime import datetime
from PIL import Image, ImageDraw

# =========================
# CONFIG
# =========================
DATA_INTERIM = Path("../../data/interim/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_512_noborder_balanced")
LABEL_STUDIO_JSON = Path("../../data/raw/SHAMPOOBLADEINTRAY_COMPLETEV2/result_labels.json")

OUT_PROCESSED = Path("../../data/processed/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
OUT_IMAGES = OUT_PROCESSED / "images"
OUT_MASKS = OUT_PROCESSED / "masks"
OUT_INDEX  = OUT_PROCESSED / "index.csv"
OUT_SPLITS = OUT_PROCESSED / "splits.json"

OUT_LABELS_DIR = Path("../../data/labels/SHAMPOOBLADEINTRAY_COMPLETEV2/gray")
OUT_LABELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TRAIN_LBL = OUT_LABELS_DIR / "train.json"
OUT_VAL_LBL   = OUT_LABELS_DIR / "val.json"
OUT_TEST_LBL  = OUT_LABELS_DIR / "test.json"

TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
SEED = 42

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}

LABEL_KEYS = [
    "Empty",
    "Non-Empty",
    "Overlap",
    "Isolated",
    "Contraband",
    "Non-Contraband",
]

OBJECT_POLYGON_LABELS = {"shampoo", "blade", "tray"}

EXTRA_NONCONTRABAND_OVERLAP_DIR = Path("../../data/interim/Stage2/gray_clahe_1500x1000_noborder_aug")

EXTRA_LABELS = {
    "Empty": 0,
    "Non-Empty": 1,
    "Overlap": 1,
    "Isolated": 0,
    "Contraband": 0,
    "Non-Contraband": 1,
}

ENABLE_EXTRA_BALANCING = True

# =========================
# HELPERS
# =========================
def list_images(root: Path):
    return sorted([
        p for p in root.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])


def norm_label(x):
    return str(x).lower().strip().replace("-", "_").replace(" ", "_")


def extract_filename_from_task(item):
    if "image" in item:
        return Path(item.get("image", "")).name

    img_ref = item.get("data", {}).get("image", "")
    return Path(img_ref).name


def parse_checkbox_labels(item):
    labels = {k: 0 for k in LABEL_KEYS}
    raw_labels = []

    # Simple format
    if "labels" in item:
        raw_labels.extend(item.get("labels", []))

    # Label Studio choice / checkbox format
    for ann in item.get("annotations", []):
        for res in ann.get("result", []):
            if res.get("type") in {"choices", "checkbox"}:
                raw_labels.extend(res.get("value", {}).get("choices", []))

    norm = {norm_label(x) for x in raw_labels}

    if "empty" in norm:
        labels["Empty"] = 1
    if "non_empty" in norm or "nonempty" in norm:
        labels["Non-Empty"] = 1
    if "overlap" in norm:
        labels["Overlap"] = 1
    if "isolated" in norm:
        labels["Isolated"] = 1
    if "contraband" in norm:
        labels["Contraband"] = 1
    if "non_contraband" in norm or "noncontraband" in norm:
        labels["Non-Contraband"] = 1

    return labels


def parse_polygon_labels(item):
    polygons = []

    for ann in item.get("annotations", []):
        for res in ann.get("result", []):
            if res.get("type", "").lower() not in {"polygonlabels", "polygon"}:
                continue

            value = res.get("value", {})
            poly_labels = value.get("polygonlabels", []) or value.get("labels", [])
            points = value.get("points", [])

            if not points:
                continue

            for lab in poly_labels:
                lab_norm = norm_label(lab)

                # Polygon labels are ONLY object masks, not classification labels
                if lab_norm not in OBJECT_POLYGON_LABELS:
                    continue

                polygons.append({
                    "label": lab_norm,
                    "points": points,
                })

    return polygons


def load_label_studio_labels(json_path: Path):
    data = json.load(open(json_path, "r"))

    labels_by_filename = {}
    polygons_by_filename = defaultdict(list)

    for item in data:
        if not isinstance(item, dict):
            continue

        fname = extract_filename_from_task(item)
        if not fname:
            continue

        # checkbox labels
        labels_by_filename[fname] = parse_checkbox_labels(item)

        # polygon masks
        polygons = parse_polygon_labels(item)
        if polygons:
            polygons_by_filename[fname].extend(polygons)

    return labels_by_filename, polygons_by_filename

def create_polygon_mask(src_path: Path, polygons):
    img = Image.open(src_path).convert("L")
    w, h = img.size

    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    for poly in polygons:
        pts = poly["points"]

        # Label Studio polygon points are in percentage coordinates
        xy = [(x / 100.0 * w, y / 100.0 * h) for x, y in pts]
        draw.polygon(xy, fill=255)

    return mask


def add_extra_noncontraband_overlap_images(image_paths, labels_by_filename, seed=42):
    if not ENABLE_EXTRA_BALANCING:
        return image_paths, labels_by_filename

    if not EXTRA_NONCONTRABAND_OVERLAP_DIR.exists():
        print("Extra folder not found:", EXTRA_NONCONTRABAND_OVERLAP_DIR)
        return image_paths, labels_by_filename

    extra_paths = list_images(EXTRA_NONCONTRABAND_OVERLAP_DIR)
    if len(extra_paths) == 0:
        print("No extra images found in:", EXTRA_NONCONTRABAND_OVERLAP_DIR)
        return image_paths, labels_by_filename

    current_counts = Counter()

    for p in image_paths:
        labels = labels_by_filename.get(p.name)

        if labels is None:
            original_name = base_group_id(p.name) + p.suffix
            labels = labels_by_filename.get(original_name)

        if labels is None:
            continue

        for k, v in labels.items():
            if v == 1:
                current_counts[k] += 1

    overlap_count = current_counts["Overlap"]
    isolated_count = current_counts["Isolated"]
    contraband_count = current_counts["Contraband"]
    non_contraband_count = current_counts["Non-Contraband"]

    need_for_overlap = max(0, isolated_count - overlap_count)
    need_for_non_contraband = max(0, contraband_count - non_contraband_count)

    num_to_add = min(
        max(need_for_overlap, need_for_non_contraband),
        len(extra_paths)
    )

    print("\nExtra balancing calculation:")
    print("  Current overlap:", overlap_count)
    print("  Current isolated:", isolated_count)
    print("  Current contraband:", contraband_count)
    print("  Current non_contraband:", non_contraband_count)
    print("  Need for overlap balance:", need_for_overlap)
    print("  Need for non_contraband balance:", need_for_non_contraband)
    print("  Extra available:", len(extra_paths))
    print("  Extra selected:", num_to_add)

    rng = random.Random(seed)
    rng.shuffle(extra_paths)
    selected_extra = extra_paths[:num_to_add]

    for p in selected_extra:
        labels_by_filename[p.name] = dict(EXTRA_LABELS)

    return image_paths + selected_extra, labels_by_filename


def base_group_id(filename: str) -> str:
    m = re.match(r"(.+?)_(orig|balflip\d+)\.[^.]+$", filename)
    if m:
        return m.group(1)
    return Path(filename).stem


def extract_timestamp(filename: str) -> str:
    m = re.search(r"(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})-(\d{3})", filename)
    if not m:
        return ""
    return f"{m.group(1)}T{m.group(2).replace('-', ':')}.{m.group(3)}"


def split_grouped_images(image_paths, seed=42):
    groups = defaultdict(list)

    for p in image_paths:
        gid = base_group_id(p.name)
        groups[gid].append(p)

    group_ids = sorted(groups.keys())
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    n_groups = len(group_ids)
    n_train = int(n_groups * TRAIN_RATIO)
    n_val = int(n_groups * VAL_RATIO)

    train_g = set(group_ids[:n_train])
    val_g = set(group_ids[n_train:n_train + n_val])
    test_g = set(group_ids[n_train + n_val:])

    rows = []

    for gid, plist in groups.items():
        if gid in train_g:
            split = "train"
        elif gid in val_g:
            split = "val"
        else:
            split = "test"

        for p in plist:
            rows.append({
                "split": split,
                "src_path": str(p),
                "filename": p.name,
                "group_id": gid,
                "scan_timestamp": extract_timestamp(p.name),
            })

    return rows


def write_outputs(rows, labels_by_filename, polygons_by_filename):
    for sp in ("train", "val", "test"):
        (OUT_IMAGES / sp).mkdir(parents=True, exist_ok=True)
        (OUT_MASKS / sp).mkdir(parents=True, exist_ok=True)

    final_rows = []
    splits_json = {"train": [], "val": [], "test": []}
    label_items = {"train": [], "val": [], "test": []}
    missing_labels = []

    for r in rows:
        split = r["split"]
        src = Path(r["src_path"])

        new_name = src.name

        dst_rel = f"images/{split}/{new_name}"
        dst_abs = OUT_PROCESSED / dst_rel

        mask_rel = f"masks/{split}/{new_name}"
        mask_abs = OUT_PROCESSED / mask_rel

        shutil.copy2(src, dst_abs)

        # Match labels/masks using exact name first, then original base name
        original_name = base_group_id(src.name) + src.suffix

        polygon_key = src.name
        if polygon_key not in polygons_by_filename:
            polygon_key = original_name

        if polygon_key in polygons_by_filename:
            mask_img = create_polygon_mask(src, polygons_by_filename[polygon_key])
        else:
            img_tmp = Image.open(src).convert("L")
            mask_img = Image.new("L", img_tmp.size, 0)

        mask_img.save(mask_abs)

        labels = labels_by_filename.get(src.name)
        if labels is None:
            labels = labels_by_filename.get(original_name)

        if labels is None:
            labels = {k: 0 for k in LABEL_KEYS}
            missing_labels.append(src.name)

        final_rows.append({
            "filepath": dst_rel,
            "maskpath": mask_rel,
            "split": split,
            "filename": new_name,
            "scan_timestamp": r["scan_timestamp"],
            "source_interim_path": str(src),
            "group_id": r["group_id"],

            "empty": labels["Empty"],
            "non_empty": labels["Non-Empty"],
            "overlap": labels["Overlap"],
            "isolated": labels["Isolated"],
            "contraband": labels["Contraband"],
            "non_contraband": labels["Non-Contraband"],
        })

        splits_json[split].append(dst_rel)

        label_items[split].append({
            "image": dst_rel,
            "mask": mask_rel,
            "empty": labels["Empty"],
            "non_empty": labels["Non-Empty"],
            "overlap": labels["Overlap"],
            "isolated": labels["Isolated"],
            "contraband": labels["Contraband"],
            "non_contraband": labels["Non-Contraband"],
        })

    with open(OUT_SPLITS, "w") as f:
        json.dump({
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "source_images": str(DATA_INTERIM),
            "source_labels": str(LABEL_STUDIO_JSON),
            "seed": SEED,
            "ratios": {
                "train": TRAIN_RATIO,
                "val": VAL_RATIO,
                "test": TEST_RATIO,
            },
            "counts": {k: len(v) for k, v in splits_json.items()},
            "missing_label_count": len(missing_labels),
            "missing_labels": missing_labels[:50],
            "splits": splits_json,
        }, f, indent=2)

    if not final_rows:
        raise RuntimeError("No rows were written. Check DATA_INTERIM and label JSON.")

    with open(OUT_INDEX, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
        w.writeheader()
        w.writerows(final_rows)

    json.dump(label_items["train"], open(OUT_TRAIN_LBL, "w"), indent=2)
    json.dump(label_items["val"], open(OUT_VAL_LBL, "w"), indent=2)
    json.dump(label_items["test"], open(OUT_TEST_LBL, "w"), indent=2)

    return final_rows, splits_json, label_items, missing_labels


# =========================
# RUN
# =========================
OUT_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Loading Label Studio JSON...")
labels_by_filename, polygons_by_filename = load_label_studio_labels(LABEL_STUDIO_JSON)
print("Loaded checkbox labels for:", len(labels_by_filename), "images")
print("Loaded polygon masks for:", len(polygons_by_filename), "images")

print("Listing images...")
image_paths = list_images(DATA_INTERIM)
print("Found original images:", len(image_paths))

image_paths, labels_by_filename = add_extra_noncontraband_overlap_images(
    image_paths,
    labels_by_filename,
    seed=SEED,
)

print("Found images after extra balancing:", len(image_paths))

rows = split_grouped_images(image_paths, seed=SEED)

final_rows, splits_json, label_items, missing_labels = write_outputs(
    rows,
    labels_by_filename,
    polygons_by_filename,
)

print("Saved processed dataset to:", OUT_PROCESSED)
print("Index saved to:", OUT_INDEX)
print("Splits saved to:", OUT_SPLITS)

print("Label files:")
print(" ", OUT_TRAIN_LBL)
print(" ", OUT_VAL_LBL)
print(" ", OUT_TEST_LBL)

print("Split counts:", {k: len(v) for k, v in splits_json.items()})
print("Missing label count:", len(missing_labels))

for split in ("train", "val", "test"):
    print(f"\n{split} label distribution:")
    for key in ["empty", "non_empty", "overlap", "isolated", "contraband", "non_contraband"]:
        print(f"  {key}: {sum(x[key] for x in label_items[split])}")